In [6]:
import numpy as np
from matplotlib import pyplot as plt

In [7]:
PATHS = {
    "train": "../data/UCI-HAR/train/",
    "test": "../data/UCI-HAR/test/"
}
NUM_CONCEPTS = 2
WINDOW_TIMESTEPS = 128


def clear_concepts (type):
    # Clearing the previous content of the concept files (if any)
    concepts_file = open("{0}concepts_{1}.txt".format(PATHS[type], type), "w")
    concepts_file.write("")
    concepts_file.close()


def concept_labeling (type):
    concepts_file = open("{0}concepts_{1}.txt".format(PATHS[type], type), "a")

    activities = np.loadtxt("{0}y_{1}.txt".format(PATHS[type], type))
    subjects = np.loadtxt("{0}subject_{1}.txt".format(PATHS[type], type))
    tot_acc_x = np.loadtxt("{0}Inertial Signals/total_acc_x_{1}.txt".format(PATHS[type], type))
    tot_acc_y = np.loadtxt("{0}Inertial Signals/total_acc_y_{1}.txt".format(PATHS[type], type))
    tot_acc_z = np.loadtxt("{0}Inertial Signals/total_acc_z_{1}.txt".format(PATHS[type], type))

    concepts = np.empty((len(activities), NUM_CONCEPTS))

    for idx in range(0, len(concepts)):
        # Labeling dynamic (1) or static (0) for classes [1, 3] and [4, 6]
        if int(activities[idx]) <= 3:
            concepts[idx][0] = 1
        else:
            concepts[idx][0] = 0

        """# Labeling samples that represent a transition between different activities (1) or not (0)
        transition = 0
        if idx > 0 and subjects[idx - 1] == subjects[idx] and activities[idx - 1] != activities[idx]:
            transition = 1
        if idx < len(activities) - 1 and subjects[idx + 1] == subjects[idx] and activities[idx + 1] != activities[idx]:
            transition = 1

        concepts[idx][1] = transition"""

        # Labeling horizontal posture (0, lying) or vertical posture (1, all the others) 
        x_sum = 0
        y_sum = 0
        z_sum = 0
        for timestep in range(0, WINDOW_TIMESTEPS):
            x_sum += tot_acc_x[idx][timestep]
            y_sum += tot_acc_y[idx][timestep]
            z_sum += tot_acc_z[idx][timestep]

        if y_sum < x_sum and z_sum < x_sum:
            concepts[idx][1] = 1
        else:
            concepts[idx][1] = 0


    # The calculated concepts are written on the related txt file
    for idx in range(0, len(concepts)):
        for concept in concepts[idx]:
            concepts_file.write(str(concept) + "  ")
        concepts_file.write("\n")

    concepts_file.close()

    

In [8]:
clear_concepts("train")
clear_concepts("test")

concept_labeling("train")
concept_labeling("test")